<a href="https://colab.research.google.com/github/DeepthiManthapuram/RAG/blob/main/chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data

In [1]:
text = """
Employees receive 12 casual leaves annually.
Employees can work from home twice per week.
Travel expenses are reimbursed within 30 days.
Medical insurance is provided to all employees.
"""


# Fixed size chunking

In [2]:
chunk_size = 50
chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

for chunk in chunks:
  print(chunk)
  print('-'*50)


Employees receive 12 casual leaves annually.
Empl
--------------------------------------------------
oyees can work from home twice per week.
Travel ex
--------------------------------------------------
penses are reimbursed within 30 days.
Medical insu
--------------------------------------------------
rance is provided to all employees.

--------------------------------------------------


# Recursive chunking

In [3]:
!pip install -q langchain-text-splitters

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 50,
    chunk_overlap=10,
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
  print(f'Chunk {i+1}')
  print(chunk)
  print('-'*50)

Chunk 1
Employees receive 12 casual leaves annually.
--------------------------------------------------
Chunk 2
Employees can work from home twice per week.
--------------------------------------------------
Chunk 3
Travel expenses are reimbursed within 30 days.
--------------------------------------------------
Chunk 4
Medical insurance is provided to all employees.
--------------------------------------------------


# Basic sentence chunking using NLTK

In [5]:
!pip install nltk
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [6]:
from nltk.tokenize import sent_tokenize

#split into sentences
chunks = sent_tokenize(text)

for i,chunk in enumerate(chunks):
  print(f'Chunk {i+1}:{chunk}')

Chunk 1:
Employees receive 12 casual leaves annually.
Chunk 2:Employees can work from home twice per week.
Chunk 3:Travel expenses are reimbursed within 30 days.
Chunk 4:Medical insurance is provided to all employees.


# Group 2 sentences per chunk

In [7]:
sentences = sent_tokenize(text)
chunk_size = 2
chunks=[]

for i in range(0, len(sentences),chunk_size):
  chunk = ' '.join(sentences[i:i+chunk_size])
  chunks.append(chunk)

for id, chunk in enumerate(chunks, start=1):
  print(f'\nChunk {id}:')
  print(chunk)


Chunk 1:

Employees receive 12 casual leaves annually. Employees can work from home twice per week.

Chunk 2:
Travel expenses are reimbursed within 30 days. Medical insurance is provided to all employees.


# Semantic Chunking Using Sentence Transformers

In [8]:
!pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sentences = sent_tokenize(text)

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(sentences)

threshold = 0.5

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
  similarity = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
  if similarity > threshold:
    current_chunk.append(sentences[i])
  else:
    chunks.append(''.join(current_chunk))
    current_chunk = sentences[i]

chunks.append(' '.join(current_chunk))

for i,chunk in enumerate(chunks, start=1):
  print(f'\nChunk {id}:')
  print(chunk)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Chunk 2:

Employees receive 12 casual leaves annually.

Chunk 2:
Employees can work from home twice per week.

Chunk 2:
Travel expenses are reimbursed within 30 days.

Chunk 2:
M e d i c a l   i n s u r a n c e   i s   p r o v i d e d   t o   a l l   e m p l o y e e s .
